In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
SELECT * from bronze_reviews

In [0]:
%sql
--CREATE OR REPLACE TABLE silver_reviews AS
WITH extracted_year AS (
  SELECT
    *,
    CAST(NULLIF(REGEXP_EXTRACT(review, '^(\\d{4})', 1), '') AS INT) as review_year
  FROM bronze_reviews
),

sample AS (
  SELECT   
    REGEXP_REPLACE(review, '^\\d{4}\\s', '') as review_text,
    CAST(ROUND(REPLACE(hours_played, ',', '')) AS INT) as hours_played,
    TRY_CAST(REPLACE(helpful, ',', '') AS INT) as helpful,
    TRY_CAST(REPLACE(funny, ',', '') AS INT) as funny,
    recommendation,
    date,
    CASE
      WHEN TRY_TO_DATE(date, 'MMMM dd, yyyy') IS NOT NULL 
        THEN TRY_TO_DATE(date, 'MMMM dd, yyyy')
      WHEN TRY_TO_DATE(date, 'MMMM d, yyyy') IS NOT NULL 
        THEN TRY_TO_DATE(date, 'MMMM d, yyyy')
      WHEN TRY_TO_DATE(date, 'd MMMM, yyyy') IS NOT NULL 
        THEN TRY_TO_DATE(date, 'd MMMM, yyyy')
      ELSE NULL
    END as review_date,
    CASE
      WHEN TRY_TO_DATE(date, 'MMMM dd, yyyy') IS NOT NULL 
        OR TRY_TO_DATE(date, 'MMMM d, yyyy') IS NOT NULL 
        OR TRY_TO_DATE(date, 'd MMMM, yyyy') IS NOT NULL THEN 0
      ELSE 1
    END as date_unparseable,
    review_year,
    game_name,
    NULLIF(TRIM(REGEXP_EXTRACT(username, '^([^\n]+)')), '') as username,
    CAST(NULLIF(REGEXP_EXTRACT(username, '(\\d+) products'), '') AS INT) as reviewer_products
  FROM extracted_year
  WHERE review IS NOT NULL AND game_name IS NOT NULL
)

SELECT *
FROM sample
WHERE TRIM(review_text) IS NOT NULL;

In [0]:
%sql
WITH sample AS (
  SELECT   
    review,
    REGEXP_REPLACE(review, '^\\d{4}\\s', '') as review_text,
    CAST(ROUND(REPLACE(hours_played, ',', '')) AS INT) as hours_played,
    TRY_CAST(REPLACE(helpful, ',', '') AS INT) as helpful,
    TRY_CAST(REPLACE(funny, ',', '') AS INT) as funny,
    recommendation,
    date,
    CASE
      WHEN TRY_TO_DATE(date, 'MMMM dd, yyyy') IS NOT NULL 
        THEN TRY_TO_DATE(date, 'MMMM dd, yyyy')
      WHEN TRY_TO_DATE(date, 'MMMM d, yyyy') IS NOT NULL 
        THEN TRY_TO_DATE(date, 'MMMM d, yyyy')
      WHEN TRY_TO_DATE(date, 'd MMMM, yyyy') IS NOT NULL 
        THEN TRY_TO_DATE(date, 'd MMMM, yyyy')
      ELSE NULL
    END as review_date,
    CASE
      WHEN TRY_TO_DATE(date, 'MMMM dd, yyyy') IS NOT NULL 
        OR TRY_TO_DATE(date, 'MMMM d, yyyy') IS NOT NULL 
        OR TRY_TO_DATE(date, 'd MMMM, yyyy') IS NOT NULL THEN 0
      ELSE 1  -- No year found
    END as date_unparseable,
    game_name,
    SUBSTRING(username, 1, POSITION(chr(10) IN username) - 1) as username,
    CAST(NULLIF(REGEXP_EXTRACT(username, '(\\d+) products'), '') AS INT) as reviewer_products
  FROM bronze_reviews
  WHERE review IS NOT NULL AND game_name IS NOT NULL
)

SELECT * FROM sample
WHERE TRIM(review_text) IS NOT NULL 
  AND TRIM(review_text) != ''

In [0]:
%sql
SELECT * FROM silver_reviews